# 1. Freezing Weights in PyTorch

Freezing controls which parameters participate in training via `requires_grad`. The backprop skips frozen weights entirely — no gradient computed, no memory allocated for optimizer state.


#### Inspecting Parameters

Always inspect before freezing — names depend on how the model was defined:

```python
for name, param in model.named_parameters():
    print(f'{name}: shape={param.shape}, trainable={param.requires_grad}')
```

`nn.Sequential` names by index (`0.weight`, `1.weight`). `nn.Module` subclasses name by attribute (`layer1.weight`, `classifier.bias`).


## 1. Full Freeze — Train Only the Head

Freeze everything, unfreeze the last layer. Use when the task is similar to pretraining and data is scarce.

```python
# freeze all
for param in model.parameters():
    param.requires_grad = False

# unfreeze last layer (Sequential)
for param in model[-1].parameters():
    param.requires_grad = True

# or by name (Module subclass)
for name, param in model.named_parameters():
    if 'classifier' in name:
        param.requires_grad = True
```

---

## 2. Partial Freeze — Train Specific Layers

Freeze early layers, train the rest. Use when the task is different enough that re-specializing only the head isn't sufficient.

```python
for param in model.parameters():
    param.requires_grad = False

for name, param in model.named_parameters():
    if any(layer in name for layer in ['layer.10', 'layer.11', 'classifier']):
        param.requires_grad = True
```

# Optimizer Memory in PyTorch

Stateful optimizers allocate tensors per parameter at the first `.step()` call. This happens regardless of `requires_grad` — the optimizer doesn't check, it just allocates for everything you passed in.

---

## What gets allocated

AdamW stores 3 tensors per parameter:

```
p  — the weight itself
m  — first moment  (exponential moving average of gradients)
v  — second moment (exponential moving average of squared gradients)

```

Update rule:

```
m = β1 * m + (1 - β1) * grad       # β1 = 0.9
v = β2 * v + (1 - β2) * grad²      # β2 = 0.999
w = w - lr * m / (√v + ε)

```

`m` and `v` are fp32 tensors — 4 bytes each per parameter.

---

## The problem

If you pass all parameters, the optimizer allocates `m` and `v` for all of them — including frozen ones that will never be updated because their `grad` is `None`.

```python
# fine-tuning a 7B model with LoRA — 99.94% of params are frozen
optimizer = AdamW(model.parameters(), lr=1e-4)
```

What gets allocated:

```
7B params × 2 tensors × 4 bytes = 56 GB of optimizer state
```

The frozen params waste 56 GB of VRAM with tensors that are never read or written.

---

## The fix

Filter to only trainable parameters before passing to the optimizer:

```python
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

```

Now only the trainable parameters get optimizer state:

```
4M params × 2 tensors × 4 bytes = 32 MB
```

---

## Which optimizers have this problem

| Optimizer | State per param | Tensors |
|---|---|---|
| AdamW / Adam | `m`, `v` | 2× |
| RMSprop | `v` | 1× |
| Adagrad | accumulator | 1× |
| SGD + momentum | `m` | 1× |
| SGD (no momentum) | none | 0× — no problem |

SGD without momentum is the only one with no extra state. In practice nobody uses it for LLMs — Adam variants dominate because of per-parameter adaptive learning rates.

In [ ]:
import torch.nn as nn

class Model(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.model = nn.Sequential(
            # FC1
            nn.Linear(10, 20, bias = True),  # MODULE 0
            nn.ReLU(), # MODULE 1
            nn.Dropout(0.2), # MODULE 2

            # FC2
            nn.Linear(20, 40, bias = True), # MODULE 3
            nn.ReLU(), # MODULE 4
            nn.Dropout(0.2), # MODULE 5

            # FC3
            nn.Linear(40, 1, bias = True),  # MODULE 6
            nn.Sigmoid(), # MODULE 7
        )

    def forward(self, x):
        x = self.model(x)
        return x
    
model = Model()


In [ ]:
# See the params and the params that have req_grad (all now)
print(f'=' * 50)
print(f'Before the freezing the weights:')
for name, param in model.named_parameters():
    print(f'Name: {name}, grad: {param.requires_grad}')

"""
Let's supose that our model have 3 layers to predict if a fish or human

1. Layer: Find characteristics in general (speak, walk in 2 foots and more)
2. Layer: Find characteristics in general (ears size, size, weight and more)
3. Layer:  Find characteristics that difference both (a main characteristics, like if have advanced intelligence)

But now we want to change the predicts, to difference if is a cat or dog, we don't need to create other model, we can just fine tunning here, we can take advantage of the layers already created and their characteristics (ears size, size, weight and more) and only change the main that it's the last layer, that difference both
"""


for name, param in model.named_parameters():
    if '.6.' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

print(f'=' * 50)
print(f'After freezing the weights')
for name, param in model.named_parameters():
    print(f'Name: {name}, grad: {param.requires_grad}')


"""
Now we can fine tunning the model, and only the last layer will change the weights

"""

Before the freezing the weights:
Name: model.0.weight, grad: False
Name: model.0.bias, grad: False
Name: model.3.weight, grad: False
Name: model.3.bias, grad: False
Name: model.6.weight, grad: True
Name: model.6.bias, grad: True
After freezing the weights
Name: model.0.weight, grad: False
Name: model.0.bias, grad: False
Name: model.3.weight, grad: False
Name: model.3.bias, grad: False
Name: model.6.weight, grad: True
Name: model.6.bias, grad: True


# Layer Replacement in PyTorch

Replace a layer by reassigning the attribute or index. The model is a Python object — reassigning is enough. The new layer initializes with random weights and `requires_grad=True` by default.

Always create the optimizer **after** replacement.

---

## 1. In a class (nn.Module)

### 1.1 Without Sequential

Access by attribute name:

```python
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 20)
        self.fc2 = nn.Linear(20, 1)

model = Model()

model.fc2 = nn.Linear(20, 5)  # replace by attribute
```

### 1.2 With Sequential inside a class

Chain attribute + index:

```python
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(10, 20),  # 0
            nn.ReLU(),          # 1
            nn.Linear(20, 1),   # 2
        )

model = Model()

model.model[2] = nn.Linear(20, 5)  # attribute + index
```

---

## 2. Without a class

### 2.1 With Sequential

Access directly by index:

```python
model = nn.Sequential(
    nn.Linear(10, 20),  # 0
    nn.ReLU(),          # 1
    nn.Linear(20, 1),   # 2
)

model[2] = nn.Linear(20, 5)  # replace by index
```



In [ ]:
print(f'Before the model changes: {model}')


"""
And now let's supose that we gonna try to predict if as a fish, human and now we add other class like car, so we need to change the last layer

1. Instead of only 1 output, we need to get 3 output (so change the Linear from 1 to 3)
2. Instead use the sigmoid we gonna use the relu

"""

model.model[6] = nn.Linear(40, 3)
model.model[7] = nn.ReLU()

print(f'=' * 50)
print(f'After the model changes: {model}')


Before the model changes: Model(
  (model): Sequential(
    (0): Linear(in_features=10, out_features=20, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=20, out_features=40, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=40, out_features=1, bias=True)
    (7): Sigmoid()
  )
)
After the model changes: Model(
  (model): Sequential(
    (0): Linear(in_features=10, out_features=20, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=20, out_features=40, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=40, out_features=3, bias=True)
    (7): ReLU()
  )
)


: 